# EnumOutputParser → Enum / `Literal` 타입 필드

`EnumOutputParser`(`langchain.output_parsers.enum`)는 LangChain v1에서 **`langchain-classic`** 으로 이동한 레거시 API입니다.

현재는 Pydantic 스키마의 필드 타입을 `Enum` 또는 `Literal`로 지정하고 `with_structured_output()`을 사용합니다.
- 허용 값이 JSON Schema의 `enum`으로 전달되고, OpenAI strict 모드에서는 **목록 밖의 값이 생성될 수 없습니다.**
- 결과는 Pydantic이 Enum 멤버로 변환해 주므로 타입 안전성도 그대로 유지됩니다.
- 색깔 외에 "이유" 같은 필드를 함께 받을 수 있습니다.

In [ ]:
# 최초 1회 설치 (LangChain v1 기준)
# %pip install -qU langchain langchain-openai langchain-classic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 의 OPENAI_API_KEY, LANGSMITH_API_KEY 를 불러옵니다.

# LangSmith 추적: 별도 헬퍼 없이 환경변수만 설정하면 자동으로 활성화됩니다.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "CH03-OutputParser")

In [ ]:
from langchain.chat_models import init_chat_model

# 공급자 중립적인 모델 초기화 ("공급자:모델명")
# 다른 모델로 바꾸려면 문자열만 교체하면 됩니다. 예) "anthropic:claude-sonnet-4-5", "ollama:llama3.1"
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

## 1. Enum 정의 (책과 동일)

In [ ]:
from enum import Enum


class Colors(Enum):
    RED = "빨간색"
    GREEN = "초록색"
    BLUE = "파란색"

## 2. Enum 필드를 가진 스키마로 구조화 출력

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class ColorAnswer(BaseModel):
    """물체의 색깔 분류 결과"""

    color: Colors = Field(description="물체의 대표 색깔")
    reason: str = Field(description="그렇게 판단한 짧은 이유")


prompt = ChatPromptTemplate.from_template("다음의 물체는 어떤 색깔인가요?\n\nObject: {object}")

chain = prompt | llm.with_structured_output(ColorAnswer)

response = chain.invoke({"object": "하늘"})
print(response)

In [ ]:
# 타입과 값을 확인합니다.
print(type(response.color))   # <enum 'Colors'>
print(response.color)         # Colors.BLUE
print(response.color.value)   # 파란색

Enum 멤버만 필요하다면 체인 끝에 lambda를 붙여 꺼낼 수 있습니다. 여러 물체는 `batch()`로 한 번에 분류합니다.

In [ ]:
color_only = chain | (lambda r: r.color)

color_only.batch([{"object": "하늘"}, {"object": "잔디"}, {"object": "딸기"}])

## 3. 별도 Enum 클래스 없이: `Literal`

값 목록만 제한하면 되는 간단한 분류에는 `Literal`이 더 간결합니다. 결과는 문자열입니다.

In [ ]:
from typing import Literal


class SimpleColor(BaseModel):
    color: Literal["빨간색", "초록색", "파란색"]


(prompt | llm.with_structured_output(SimpleColor)).invoke({"object": "소방차"})

## (참고) 레거시 API

`from langchain_classic.output_parsers.enum import EnumOutputParser`로 여전히 사용할 수 있지만, 신규 코드에는 권장하지 않습니다.